In [1]:
import random
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torchvision.models as models
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, Resize, ToTensor
from tqdm import tqdm

In [2]:
SEED = 492
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
data_path = Path("../../data/ISIC")
image_path = data_path / "ISIC_2024_Training_Input"
ground_truth_path = data_path / "ISIC_2024_Training_GroundTruth.csv"
metadata_path = data_path / "metadata.csv"
model_path = Path("../../models")

In [4]:
device = torch.device("cuda")

In [5]:
ground_truth_df = pl.read_csv(ground_truth_path)
metadata_df = pl.read_csv(metadata_path)

df = ground_truth_df.join(metadata_df, on="isic_id", how="inner").with_columns(
    (pl.lit(str(image_path)) + "/" + pl.col("isic_id").cast(pl.Utf8) + ".jpg").alias("image_path")
)

patient_stats = df.group_by("patient_id").agg(pl.col("malignant").max().alias("has_malignant")).sort("patient_id")

train_p, val_p = train_test_split(
    patient_stats, test_size=0.1, random_state=SEED, stratify=patient_stats["has_malignant"]
)

train_ids = train_p["patient_id"].to_list()
val_ids = val_p["patient_id"].to_list()

train_df = df.filter(pl.col("patient_id").is_in(train_ids))
val_df = df.filter(pl.col("patient_id").is_in(val_ids))

In [6]:
tab_categorical = ["sex", "anatom_site_general"]
tab_numerical = ["age_approx", "clin_size_long_diam_mm", "tbp_lv_areaMM2", "tbp_lv_eccentricity"]

median_age = train_df["age_approx"].median()
mode_sex = train_df["sex"].drop_nulls().mode()[0]

train_df = train_df.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown")
)

val_df = val_df.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown")
)

exprs = []
for c in tab_numerical:
    exprs.append(pl.col(c).mean().alias(f"{c}_mean"))
    exprs.append(pl.col(c).std().alias(f"{c}_std"))

num_stats = train_df.select(exprs)

for col in tab_numerical:
    mean = num_stats.item(0, f"{col}_mean")
    std = num_stats.item(0, f"{col}_std")
    val_df = val_df.with_columns(((pl.col(col) - mean) / std).alias(col))

tab_features = list(tab_numerical)

for col in tab_categorical:
    categories = train_df[col].unique().to_list()
    for cat in categories:
        col_name = f"{col}_{cat}"
        val_df = val_df.with_columns((pl.col(col) == cat).cast(pl.Int8).alias(col_name))
        tab_features.append(col_name)

val_df = val_df.drop(tab_categorical)

In [7]:
class MultimodalModel(nn.Module):
    def __init__(self, tab_dim):
        super().__init__()
        self.image_encoder = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.image_encoder.fc = nn.Identity()

        self.tab_encoder = nn.Sequential(
            nn.Linear(tab_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )

        self.fusion_head = nn.Sequential(
            nn.Linear(2048 + 32, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, image, tab):
        img_feat = self.image_encoder(image)
        tab_feat = self.tab_encoder(tab)
        combined = torch.cat([img_feat, tab_feat], dim=1)
        return self.fusion_head(combined)

In [8]:
class ISICMultimodalDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame, tabular_features: list[str], image_transform):
        self.df = dataframe
        self.tab_features = tabular_features
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)
        img = Image.open(row["image_path"]).convert("RGB")
        img = self.image_transform(img)
        tab_data = torch.tensor([row[feat] for feat in self.tab_features], dtype=torch.float32)
        label = torch.tensor(row["malignant"], dtype=torch.float32)
        return img, tab_data, label

In [9]:
val_transform = Compose([Resize((224, 224)), ToTensor()])
val_dataset = ISICMultimodalDataset(val_df, tab_features, val_transform)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

In [10]:
model = MultimodalModel(tab_dim=len(tab_features)).to(device)
model.load_state_dict(torch.load(model_path / "best_hard_neg_model.pt", weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, tabs, labels in tqdm(val_loader, desc="Inference"):
        images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images, tabs)
        all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

Inference: 100%|██████████| 376/376 [01:18<00:00,  4.81it/s]


In [19]:
desc_score_indices = np.argsort(all_preds, kind="mergesort")[::-1]
y_score = all_preds[desc_score_indices]
y_true = all_labels[desc_score_indices]

distinct_value_indices = np.where(np.diff(y_score))[0]
threshold_indices = np.concatenate([distinct_value_indices, np.array([len(y_true) - 1])])

tps = np.cumsum(y_true)[threshold_indices]
fps = threshold_indices + 1 - tps

total_pos = tps[-1]

recalls = tps / total_pos
precisions = tps / (tps + fps)
thresholds = y_score[threshold_indices]

pr_df = pl.DataFrame({
    "threshold": thresholds,
    "precision": precisions,
    "recall": recalls,
})

results = []

for tr in [0.50, 0.75, 0.90, 0.95, 0.99]:
    valid = pr_df.filter(pl.col("recall") >= tr - 1e-9)
    if valid.height > 0:
        best = valid.sort("precision", descending=True)
        results.append({
            "Metric": f"Recall >= {int(tr*100)}%",
            "Threshold": best["threshold"][0],
            "Precision": best["precision"][0],
            "Recall": best["recall"][0],
        })

if results:
    final_df = pl.DataFrame(results).sort("Threshold", descending=True)

    def fmt_dec(x):
        if x is None or (isinstance(x, float) and np.isnan(x)): return "N/A"
        return f"{x:.4f}"

    def fmt_pct(x):
        if x is None or (isinstance(x, float) and np.isnan(x)): return "N/A"
        return f"{x:.2%}"

    final_df = final_df.with_columns([
        pl.col("Threshold").map_elements(fmt_dec, return_dtype=pl.Utf8),
        pl.col("Precision").map_elements(fmt_pct, return_dtype=pl.Utf8),
        pl.col("Recall").map_elements(fmt_pct, return_dtype=pl.Utf8),
    ])

    display(final_df)

Metric,Threshold,Precision,Recall
str,str,str,str
"""Recall >= 50%""","""0.9023""","""1.85%""","""51.11%"""
"""Recall >= 75%""","""0.5884""","""0.89%""","""75.56%"""
"""Recall >= 90%""","""0.1351""","""0.35%""","""91.11%"""
"""Recall >= 95%""","""0.0069""","""0.14%""","""97.78%"""
"""Recall >= 99%""","""0.0045""","""0.14%""","""100.00%"""
